In [6]:
import os
import shutil

import torch
from safetensors.torch import load_file
from huggingface_hub import snapshot_download 

In [7]:
from nicheformer.models import Nicheformer

repo_dir = snapshot_download(
    repo_id="theislab/Nicheformer",
    allow_patterns=[
        "config.json",
        "model.safetensors",
        "configuration_nicheformer.py",
        "modeling_nicheformer.py"
    ],
)

# here are the 
# * configuration_nicheformer.py
# * modeling_nicheformer.py
# files, install them to the model/ directory so that we can import them
if not os.path.exists("model"):
    os.makedirs("model")
shutil.copy(f"{repo_dir}/configuration_nicheformer.py", "model/")
shutil.copy(f"{repo_dir}/modeling_nicheformer.py", "model/")

Fetching 4 files: 100%|██████████| 4/4 [00:00<00:00, 3000.75it/s]


'model/modeling_nicheformer.py'

In [9]:
# get configuration and model from the downloaded files
from model.configuration_nicheformer import NicheformerConfig
from model.modeling_nicheformer import NicheformerForMaskedLM

# define a scriptable wrapper that reproduces get_embeddings
class NicheformerTorchscript(torch.nn.Module):
    def __init__(self, model: torch.nn.Module):
        super().__init__()
        core = model.nicheformer  # NicheformerModel inside NicheformerForMaskedLM
        if not isinstance(core.positional_embedding, torch.nn.Embedding):
            raise ValueError("expected learnable positional embeddings")

        # only what the embedding path needs: submodules, a buffer and ints
        self.embeddings = core.embeddings
        self.positional_embedding = core.positional_embedding
        self.register_buffer("pos", core.pos.clone())
        self.layers = core.encoder.layers
        self.n_layers = len(core.encoder.layers)
        self.n_context_tokens = 3  # modality, species and assay tokens

    def forward(
        self,
        input_ids: torch.Tensor,
        attention_mask: torch.Tensor,
        layer: int = -1,
        with_context: bool = False,
    ) -> torch.Tensor:
        
        last = layer if layer >= 0 else self.n_layers + layer
        if last < 0 or last >= self.n_layers:
            raise ValueError("layer is out of range")
        # dropout is a no-op in eval mode, so it is left out
        x = self.embeddings(input_ids) + self.positional_embedding(self.pos[: input_ids.size(1)])
        padding_mask = attention_mask == 0

        # TorchScript can't index a ModuleList with a runtime int, so loop and skip
        i = 0
        for block in self.layers:
            if i <= last:
                x = block(x, src_key_padding_mask=padding_mask)
            i += 1
        start = 0 if with_context else self.n_context_tokens
        return x[:, start:, :].mean(dim=1)

# configure the model and load the weights
config = NicheformerConfig.from_pretrained(repo_dir)
hf_model = NicheformerForMaskedLM(config)

state = load_file(f"{repo_dir}/model.safetensors")
missing, unexpected = hf_model.load_state_dict(state, strict=False)

print("missing:", missing)
print("unexpected:", unexpected)

hf_model.eval()
hf_model.requires_grad_(False)

wrapper = NicheformerTorchscript(hf_model).eval()

# two cells, the second with a padded tail, so the attention mask is exercised
torch.manual_seed(0)
n_tokens = wrapper.embeddings.num_embeddings
context_length = wrapper.pos.numel()
dummy_input_ids = torch.randint(0, n_tokens, (2, context_length), dtype=torch.long)
dummy_attention_mask = torch.ones((2, context_length), dtype=torch.long)
dummy_attention_mask[1, 900:] = 0

# compile the Python code; no example inputs needed
scripted = torch.jit.script(wrapper)
scripted.eval()

with torch.inference_mode():
    # reference: the original Hugging Face method
    eager_out = hf_model.get_embeddings(
        input_ids=dummy_input_ids,
        attention_mask=dummy_attention_mask,
        layer=-1,
        with_context=False,
    )
    scripted_out = scripted(dummy_input_ids, dummy_attention_mask)

    # layer and with_context are now runtime arguments
    eager_l5 = hf_model.get_embeddings(
        input_ids=dummy_input_ids,
        attention_mask=dummy_attention_mask,
        layer=5,
        with_context=True,
    )
    scripted_l5 = scripted(dummy_input_ids, dummy_attention_mask, 5, True)

torch.testing.assert_close(
    eager_out.detach().cpu(),
    scripted_out.detach().cpu(),
    rtol=1e-4,
    atol=1e-4,
)
torch.testing.assert_close(eager_l5, scripted_l5, rtol=1e-4, atol=1e-4)

# Save UNFROZEN (run_nicheformer.cpp lists the parameters), with metadata for C++.
scripted.save(
    "nicheformer_embeddings.pt",
    _extra_files={
        "context_length": str(context_length),
        "embedding_dim": str(wrapper.embeddings.embedding_dim),
        "n_layers": str(wrapper.n_layers),
    },
)

# Test Python reload.
reloaded = torch.jit.load("nicheformer_embeddings.pt", map_location="cpu")
reloaded.eval()

with torch.inference_mode():
    reload_out = reloaded(dummy_input_ids, dummy_attention_mask)

torch.testing.assert_close(
    eager_out.detach().cpu(),
    reload_out.detach().cpu(),
    rtol=1e-4,
    atol=1e-4,
)

print("saved: nicheformer_embeddings.pt")
print("n params:", sum(p.numel() for p in hf_model.parameters()))

missing: []
unexpected: []
saved: nicheformer_embeddings.pt
n params: 48955252
